# Sarcasm Detection on SARC — Kaggle notebook

Runs the project on Kaggle's free GPU. Kaggle is convenient because the SARC
dataset is already hosted here, so there's nothing to download.

### Before running
1. **Enable the GPU:** right sidebar → *Settings → Accelerator → GPU T4 x2* (or P100).
2. **Add the dataset:** *+ Add Data* → search **"Sarcasm"** → add **danofer/sarcasm**.
   It mounts read-only under `/kaggle/input/`.
3. **Add the code** (one of the options in the next cell): clone your GitHub repo,
   or upload the repo as a Kaggle Dataset and copy it in. (For cloning you must turn
   ON *Settings → Internet*.)

In [ ]:
# --- Get the project code into the WRITABLE working dir -------------------
# Option A: clone your GitHub repo (requires Internet ON in Settings):
# !git clone https://github.com/<you>/EmiliaProject.git /kaggle/working/EmiliaProject
#
# Option B: you uploaded the repo as a Kaggle Dataset -> copy it where we can write:
# !cp -r /kaggle/input/<your-code-dataset-slug>/EmiliaProject /kaggle/working/

import os
CODE_DIR = "/kaggle/working/EmiliaProject/project"
assert os.path.isdir(CODE_DIR), (
    "Put the repo in /kaggle/working (clone it, or copy it from an uploaded dataset)."
)
print("code at", CODE_DIR)

In [ ]:
# --- Point the pipeline at the Kaggle-mounted SARC csv --------------------
import glob, os
hits = glob.glob("/kaggle/input/**/train-balanced-sarcasm.csv", recursive=True)
assert hits, "Add the 'Sarcasm' (danofer/sarcasm) dataset via + Add Data first."
os.environ["SARC_CSV"] = hits[0]          # config.py reads this env var
print("SARC_CSV =", hits[0])

In [ ]:
# --- Make sure transformers is recent enough (needs >=4.46) --------------
# Kaggle usually ships a recent version; upgrade only if needed.
!pip install -q -U "transformers>=4.46" datasets
import torch
print("CUDA available:", torch.cuda.is_available())   # True on a GPU notebook

In [ ]:
# --- Sanity-check the data pipeline --------------------------------------
!python $CODE_DIR/run.py prepare

## Baseline (TF-IDF) — seconds

In [ ]:
!python $CODE_DIR/run.py baseline
!python $CODE_DIR/run.py baseline --context

## Fine-tune the transformers

On a CUDA GPU the code turns on fp16 automatically, so these are much faster than
on a laptop. `--subset` keeps it quick; raise it (or drop it for the full ~1M rows)
once you're happy.

In [ ]:
!python $CODE_DIR/run.py train --model bert-base-uncased            --subset 50000 --epochs 3
!python $CODE_DIR/run.py train --model bert-base-uncased --context  --subset 50000 --epochs 3

In [ ]:
!python $CODE_DIR/run.py train --model roberta-base            --subset 50000 --epochs 3
!python $CODE_DIR/run.py train --model roberta-base --context  --subset 50000 --epochs 3

In [ ]:
# --- McNemar significance test: does context help BERT? ------------------
!python $CODE_DIR/run.py compare \
    --a /kaggle/working/EmiliaProject/results/bert-base-uncased_noctx_seed42_preds.npz \
    --b /kaggle/working/EmiliaProject/results/bert-base-uncased_ctx_seed42_preds.npz

In [ ]:
# --- Aggregate everything into the comparison table ----------------------
!python $CODE_DIR/run.py report

from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/EmiliaProject/results/summary.md").read()))

### Saving your outputs
Everything is written under `/kaggle/working/EmiliaProject/results/` and `/models/`.
Use *Save Version* (top-right) to persist the notebook and its `/kaggle/working`
outputs, or download `results/summary.md` and the `*.png` confusion matrices.